# 🛠️ Pré-processamento e Feature Engineering - Predição de Limite de Crédito

Este notebook prepara a base para modelagem, sem treinar modelos.

## Objetivos
- Remover colunas irrelevantes
- Tratar duplicidades
- Criar variáveis derivadas
- Codificar variáveis categóricas
- Salvar um CSV tratado em `data/processed`

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
RAW_PATH = Path("../data/raw/Credit.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
df.head()

,index,Income,Limit,Rating,Cards,Age,Education,Gender,Student,Married,Ethnicity,Balance
0,1,14.891,3606,283,2,34,11,Male,No,Yes,Caucasian,333
1,2,106.025,6645,483,3,82,15,Female,Yes,Yes,Asian,903
2,3,104.593,7075,514,4,71,11,Male,No,No,Asian,580
3,4,148.924,9504,681,3,36,11,Female,No,No,Asian,964
4,5,55.882,4897,357,2,68,16,Male,No,Yes,Caucasian,331


## 1. Visão inicial

In [4]:
print(f"Shape inicial: {df.shape}")
df.info()

Shape inicial: (400, 12)
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   index      400 non-null    int64  
 1   Income     400 non-null    float64
 2   Limit      400 non-null    int64  
 3   Rating     400 non-null    int64  
 4   Cards      400 non-null    int64  
 5   Age        400 non-null    int64  
 6   Education  400 non-null    int64  
 7   Gender     400 non-null    str    
 8   Student    400 non-null    str    
 9   Married    400 non-null    str    
 10  Ethnicity  400 non-null    str    
 11  Balance    400 non-null    int64  
dtypes: float64(1), int64(7), str(4)
memory usage: 37.6 KB


## 2. Limpeza inicial

In [5]:
# remove coluna índice que veio da origem
if "index" in df.columns:
    df = df.drop(columns=["index"])
elif "" in df.columns:
    df = df.drop(columns=[""])

print(f"Shape após remover coluna índice: {df.shape}")

Shape após remover coluna índice: (400, 11)


In [6]:
duplicados = df.duplicated().sum()
print(f"Duplicados encontrados: {duplicados}")

if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Shape após remoção de duplicados: {df.shape}")

Duplicados encontrados: 0
Shape após remoção de duplicados: (400, 11)


## 3. Feature engineering

In [ ]:
# razões simples que podem ajudar a explicar limite concedido
if {"Balance", "Income"}.issubset(df.columns):
    df["debt_to_income"] = df["Balance"] / (df["Income"] + 1)

if {"Cards", "Age"}.issubset(df.columns):
    df["cards_per_age"] = df["Cards"] / (df["Age"] + 1)

df.head()

,Income,Limit,Rating,Cards,Age,Education,Gender,Student,Married,Ethnicity,Balance,debt_to_income,cards_per_age
0,14.891,3606,283,2,34,11,Male,No,Yes,Caucasian,333,20.955258,0.057143
1,106.025,6645,483,3,82,15,Female,Yes,Yes,Asian,903,8.437281,0.036145
2,104.593,7075,514,4,71,11,Male,No,No,Asian,580,5.492788,0.055556
3,148.924,9504,681,3,36,11,Female,No,No,Asian,964,6.429924,0.081081
4,55.882,4897,357,2,68,16,Male,No,Yes,Caucasian,331,5.819064,0.028986


## 4. Encoding das variáveis categóricas

In [8]:
target = "Limit"

categorical_cols = df.drop(columns=[target]).select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df.drop(columns=[target]).select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categóricas:", categorical_cols)
print("Numéricas:", numerical_cols)

Categóricas: ['Gender', 'Student', 'Married', 'Ethnicity']
Numéricas: ['Income', 'Rating', 'Cards', 'Age', 'Education', 'Balance', 'debt_to_income', 'cards_per_age']


C:\Users\guilh\AppData\Local\Temp\ipykernel_21464\489840613.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.drop(columns=[target]).select_dtypes(include=["object"]).columns.tolist()


In [9]:
# one-hot encoding sem usar a target
df_processed = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape final tratada: {df_processed.shape}")
df_processed.head()

Shape final tratada: (400, 14)


,Income,Limit,Rating,Cards,Age,Education,Balance,debt_to_income,cards_per_age,Gender_Female,Student_Yes,Married_Yes,Ethnicity_Asian,Ethnicity_Caucasian
0,14.891,3606,283,2,34,11,333,20.955258,0.057143,False,False,True,False,True
1,106.025,6645,483,3,82,15,903,8.437281,0.036145,True,True,True,True,False
2,104.593,7075,514,4,71,11,580,5.492788,0.055556,False,False,False,True,False
3,148.924,9504,681,3,36,11,964,6.429924,0.081081,True,False,False,True,False
4,55.882,4897,357,2,68,16,331,5.819064,0.028986,False,False,True,False,True


## 5. Validação final

In [10]:
print(df_processed.isnull().sum().sort_values(ascending=False).head(10))

Income            0
Limit             0
Rating            0
Cards             0
Age               0
Education         0
Balance           0
debt_to_income    0
cards_per_age     0
Gender_Female     0
dtype: int64


In [11]:
print(df_processed.dtypes)

Income                 float64
Limit                    int64
Rating                   int64
Cards                    int64
Age                      int64
Education                int64
Balance                  int64
debt_to_income         float64
cards_per_age          float64
Gender_Female             bool
Student_Yes               bool
Married_Yes               bool
Ethnicity_Asian           bool
Ethnicity_Caucasian       bool
dtype: object


## 6. Salvar base tratada

In [13]:
OUTPUT_PATH = PROCESSED_DIR / "credit_processed.csv"
df_processed.to_csv(OUTPUT_PATH, index=False)

print(f"Shape exportado: {df_processed.shape}")

Shape exportado: (400, 14)
